# LangGraph and LangSmith - Agentic RAG Powered by LangChain

In the following notebook we'll complete the following tasks:

- 🤝 Breakout Room #1:
  1. Install required libraries
  2. Set Environment Variables
  3. Creating our Tool Belt
  4. Creating Our State
  5. Creating and Compiling A Graph!

- 🤝 Breakout Room #2:
  1. Evaluating the LangGraph Application with LangSmith
  2. Adding Helpfulness Check and "Loop" Limits
  3. LangGraph for the "Patterns" of GenAI

# 🤝 Breakout Room #1

## Part 1: LangGraph - Building Cyclic Applications with LangChain

LangGraph is a tool that leverages LangChain Expression Language to build coordinated multi-actor and stateful applications that includes cyclic behaviour.

### Why Cycles?

In essence, we can think of a cycle in our graph as a more robust and customizable loop. It allows us to keep our application agent-forward while still giving the powerful functionality of traditional loops.

Due to the inclusion of cycles over loops, we can also compose rather complex flows through our graph in a much more readable and natural fashion. Effectively allowing us to recreate application flowcharts in code in an almost 1-to-1 fashion.

### Why LangGraph?

Beyond the agent-forward approach - we can easily compose and combine traditional "DAG" (directed acyclic graph) chains with powerful cyclic behaviour due to the tight integration with LCEL. This means it's a natural extension to LangChain's core offerings!

## Task 1:  Dependencies


## Task 2: Environment Variables

We'll want to set our OpenAI, Tavily, and LangSmith API keys along with our LangSmith environment variables.

In [1]:
import os
import getpass

os.environ["OPENAI_API_KEY"] = getpass.getpass("OpenAI API Key:")

In [2]:
os.environ["TAVILY_API_KEY"] = getpass.getpass("TAVILY_API_KEY")

In [3]:
from uuid import uuid4

os.environ["LANGCHAIN_TRACING_V2"] = "true"
os.environ["LANGCHAIN_PROJECT"] = f"AIE8 - LangGraph - {uuid4().hex[0:8]}"
os.environ["LANGCHAIN_API_KEY"] = getpass.getpass("LangSmith API Key: ")

## Task 3: Creating our Tool Belt

As is usually the case, we'll want to equip our agent with a toolbelt to help answer questions and add external knowledge.

There's a tonne of tools in the [LangChain Community Repo](https://github.com/langchain-ai/langchain-community/tree/main/libs/community) but we'll stick to a couple just so we can observe the cyclic nature of LangGraph in action!

We'll leverage:

- [Tavily Search Results](https://github.com/langchain-ai/langchain-community/blob/main/libs/community/langchain_community/tools/tavily_search/tool.py)
- [Arxiv](https://github.com/langchain-ai/langchain-community/blob/main/libs/community/langchain_community/tools/arxiv/tool.py)

#### 🏗️ Activity #1:

Please add the tools to use into our toolbelt.

> NOTE: Each tool in our toolbelt should be a method.

In [4]:
from langchain_community.tools.tavily_search import TavilySearchResults
from langchain_community.tools.arxiv.tool import ArxivQueryRun

tavily_tool = TavilySearchResults(max_results=5)

tool_belt = [
    tavily_tool,
    ArxivQueryRun(),
]

/var/folders/wz/l5kzl3495bj9gj9jt4p33xwh0000gn/T/ipykernel_12979/1203815797.py:4: LangChainDeprecationWarning: The class `TavilySearchResults` was deprecated in LangChain 0.3.25 and will be removed in 1.0. An updated version of the class exists in the :class:`~langchain-tavily package and should be used instead. To use it run `pip install -U :class:`~langchain-tavily` and import as `from :class:`~langchain_tavily import TavilySearch``.
  tavily_tool = TavilySearchResults(max_results=5)


### Model

Now we can set-up our model! We'll leverage the familiar OpenAI model suite for this example - but it's not *necessary* to use with LangGraph. LangGraph supports all models - though you might not find success with smaller models - as such, they recommend you stick with:

- OpenAI's GPT-3.5 and GPT-4
- Anthropic's Claude
- Google's Gemini

> NOTE: Because we're leveraging the OpenAI function calling API - we'll need to use OpenAI *for this specific example* (or any other service that exposes an OpenAI-style function calling API.

In [5]:
from langchain_openai import ChatOpenAI

model = ChatOpenAI(model="gpt-4.1-nano", temperature=0)

Now that we have our model set-up, let's "put on the tool belt", which is to say: We'll bind our LangChain formatted tools to the model in an OpenAI function calling format.

In [7]:
model = model.bind_tools(tool_belt)

#### ❓ Question #1:

How does the model determine which tool to use?

✅ Answer:

# How a Model Determines Which Tool to Use

- **Tool Schemas**: Each tool is described with a JSON schema containing:  
  - Tool name (e.g., `"web_search"`, `"calculator"`)  
  - Description of what the tool does  
  - Parameters the tool accepts (with types and descriptions)  
  - Whether parameters are required or optional  

A model decides which tool to use by **semantically matching** the user’s request against these descriptions.  

Think of it like a **smart switchboard operator 🧠**: the operator (the model) listens to your request (the prompt), checks its directory of available departments (the tools), and connects you to the right one based on meaning—not just keywords.  

## The Core Mechanism: Semantic Matching

When tools are bound to a model, each comes with a name and detailed description. The model then:  

1. **Analyze the Prompt** → Understand the goal or question.  
   - *Example*: In *“What’s the weather in Toronto and how does that compare to the current price of Google’s stock?”*, the model detects two intents: weather and stock price.  

2. **Match to Tool Descriptions** → Align each intent with the most relevant tool.  
   - “Get weather” → *Fetches current weather for a location*  
   - “Get stock price” → *Retrieves the latest stock price for a ticker symbol*  

3. **Extract Arguments** → Pull required inputs from the prompt.  
   - For weather: `"Toronto"`  
   - For stock: `"Google"` → converted to `"GOOGL"`  

4. **Generate a Tool Call** → Produce a structured JSON request specifying the tool and its arguments.  

⚡ **Note**: The model itself doesn’t run the tool—it outputs the tool call, which your system then executes.

---

## Task 4: Putting the State in Stateful

Earlier we used this phrasing:

`coordinated multi-actor and stateful applications`

So what does that "stateful" mean?

To put it simply - we want to have some kind of object which we can pass around our application that holds information about what the current situation (state) is. Since our system will be constructed of many parts moving in a coordinated fashion - we want to be able to ensure we have some commonly understood idea of that state.

LangGraph leverages a `StatefulGraph` which uses an `AgentState` object to pass information between the various nodes of the graph.

There are more options than what we'll see below - but this `AgentState` object is one that is stored in a `TypedDict` with the key `messages` and the value is a `Sequence` of `BaseMessages` that will be appended to whenever the state changes.

Let's think about a simple example to help understand exactly what this means (we'll simplify a great deal to try and clearly communicate what state is doing):

1. We initialize our state object:
  - `{"messages" : []}`
2. Our user submits a query to our application.
  - New State: `HumanMessage(#1)`
  - `{"messages" : [HumanMessage(#1)}`
3. We pass our state object to an Agent node which is able to read the current state. It will use the last `HumanMessage` as input. It gets some kind of output which it will add to the state.
  - New State: `AgentMessage(#1, additional_kwargs {"function_call" : "WebSearchTool"})`
  - `{"messages" : [HumanMessage(#1), AgentMessage(#1, ...)]}`
4. We pass our state object to a "conditional node" (more on this later) which reads the last state to determine if we need to use a tool - which it can determine properly because of our provided object!

In [8]:
from typing import TypedDict, Annotated
from langgraph.graph.message import add_messages
import operator
from langchain_core.messages import BaseMessage

class AgentState(TypedDict):
  messages: Annotated[list, add_messages]

## Task 5: It's Graphing Time!

Now that we have state, and we have tools, and we have an LLM - we can finally start making our graph!

Let's take a second to refresh ourselves about what a graph is in this context.

Graphs, also called networks in some circles, are a collection of connected objects.

The objects in question are typically called nodes, or vertices, and the connections are called edges.

Let's look at a simple graph.

![image](https://i.imgur.com/2NFLnIc.png)

Here, we're using the coloured circles to represent the nodes and the yellow lines to represent the edges. In this case, we're looking at a fully connected graph - where each node is connected by an edge to each other node.

If we were to think about nodes in the context of LangGraph - we would think of a function, or an LCEL runnable.

If we were to think about edges in the context of LangGraph - we might think of them as "paths to take" or "where to pass our state object next".

Let's create some nodes and expand on our diagram.

> NOTE: Due to the tight integration with LCEL - we can comfortably create our nodes in an async fashion!

In [9]:
from langgraph.prebuilt import ToolNode

def call_model(state):
  messages = state["messages"]
  response = model.invoke(messages)
  return {"messages" : [response]}

tool_node = ToolNode(tool_belt)

Now we have two total nodes. We have:

- `call_model` is a node that will...well...call the model
- `tool_node` is a node which can call a tool

Let's start adding nodes! We'll update our diagram along the way to keep track of what this looks like!


In [10]:
from langgraph.graph import StateGraph, END

uncompiled_graph = StateGraph(AgentState)

uncompiled_graph.add_node("agent", call_model)
uncompiled_graph.add_node("action", tool_node)

Let's look at what we have so far:

![image](https://i.imgur.com/md7inqG.png)

Next, we'll add our entrypoint. All our entrypoint does is indicate which node is called first.

In [11]:
uncompiled_graph.set_entry_point("agent")

![image](https://i.imgur.com/wNixpJe.png)

Now we want to build a "conditional edge" which will use the output state of a node to determine which path to follow.

We can help conceptualize this by thinking of our conditional edge as a conditional in a flowchart!

Notice how our function simply checks if there is a "function_call" kwarg present.

Then we create an edge where the origin node is our agent node and our destination node is *either* the action node or the END (finish the graph).

It's important to highlight that the dictionary passed in as the third parameter (the mapping) should be created with the possible outputs of our conditional function in mind. In this case `should_continue` outputs either `"end"` or `"continue"` which are subsequently mapped to the action node or the END node.

In [12]:
def should_continue(state):
  last_message = state["messages"][-1]

  if last_message.tool_calls:
    return "action"

  return END

uncompiled_graph.add_conditional_edges(
    "agent",
    should_continue
)

Let's visualize what this looks like.

![image](https://i.imgur.com/8ZNwKI5.png)

Finally, we can add our last edge which will connect our action node to our agent node. This is because we *always* want our action node (which is used to call our tools) to return its output to our agent!

In [13]:
uncompiled_graph.add_edge("action", "agent")

Let's look at the final visualization.

![image](https://i.imgur.com/NWO7usO.png)

All that's left to do now is to compile our workflow - and we're off!

In [14]:
simple_agent_graph = uncompiled_graph.compile()

#### ❓ Question #2:

Is there any specific limit to how many times we can cycle?

If not, how could we impose a limit to the number of cycles?

✅ Answer:

### ✅ Answer

**Default Behavior**  
- Cycles are **allowed** in LangGraph (it’s a core feature).  
- Execution is **not infinite**: it is bounded by a **recursion limit**.  
- **Default limit = 25 super-steps**.  
- If exceeded, LangGraph raises a `GraphRecursionError`.  

**What counts as a “step”**  
- Each **super-step** represents one sequential pass.  
- Nodes running in **parallel** are part of the same step.  
- Nodes running **sequentially** count as separate steps.  

**Ways to Control Cycle Limits**  
1. **Global recursion limit (built-in)**  
   - Raise or lower dynamically at runtime:  
     ```python
     graph.invoke(inputs, config={"recursion_limit": 50})
     ```
   - Configure permanently with `.with_config()`.  
   - Handle exceptions safely:  
     ```python
     from langgraph.errors import GraphRecursionError
     ```

2. **Custom counters in state**  
   - Track iterations (`iteration_count`, `max_iterations`) inside the graph state.  
   - Route to `END` when threshold is reached.  

3. **Node-specific limits**  
   - Maintain a `step_counter` per node for fine-grained control over loop iterations.  

4. **Other approaches**  
   - **Time-based limits**: stop execution after a timeout.  
   - **Conditional edges**: decide dynamically whether to continue or terminate.  

**Best Practices**  
- Always set **reasonable limits** to prevent runaway loops.  
- Use **multiple termination conditions** (counter + completion condition).  
- Log cycle counts for monitoring and debugging.  
- Gracefully handle `GraphRecursionError` with fallback strategies in production applications. 

### 👉 Conclusion:
LangGraph cycles are **limited by the recursion limit (25 by default)**.  
You can increase it (e.g., `recursion_limit=100`), decrease it to stop earlier, or implement **custom state counters** and **conditional routing** for finer control.  


## Using Our Graph

Now that we've created and compiled our graph - we can call it *just as we'd call any other* `Runnable`!

Let's try out a few examples to see how it fairs:

In [15]:
from langchain_core.messages import HumanMessage

inputs = {"messages" : [HumanMessage(content="How are technical professionals using AI to improve their work?")]}

async for chunk in simple_agent_graph.astream(inputs, stream_mode="updates"):
    for node, values in chunk.items():
        print(f"Receiving update from node: '{node}'")
        print(values["messages"])
        print("\n\n")

Receiving update from node: 'agent'
[AIMessage(content='Technical professionals are using AI in various ways to enhance their work, including automating repetitive tasks, improving decision-making, analyzing large datasets, developing new products and services, and optimizing processes. They leverage AI for tasks such as machine learning model development, natural language processing, computer vision, predictive analytics, and automation. This integration helps increase efficiency, accuracy, and innovation across different industries. Would you like specific examples from particular fields or industries?', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 88, 'prompt_tokens': 163, 'total_tokens': 251, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_name': 'gpt-4.1-nano-2025-04-14',

Let's look at what happened:

1. Our state object was populated with our request
2. The state object was passed into our entry point (agent node) and the agent node added an `AIMessage` to the state object and passed it along the conditional edge
3. The conditional edge received the state object, found the "tool_calls" `additional_kwarg`, and sent the state object to the action node
4. The action node added the response from the OpenAI function calling endpoint to the state object and passed it along the edge to the agent node
5. The agent node added a response to the state object and passed it along the conditional edge
6. The conditional edge received the state object, could not find the "tool_calls" `additional_kwarg` and passed the state object to END where we see it output in the cell above!

Now let's look at an example that shows a multiple tool usage - all with the same flow!

In [16]:
inputs = {"messages" : [HumanMessage(content="Search Arxiv for the A Comprehensive Survey of Deep Research paper, then search each of the authors to find out where they work now using Tavily!")]}

async for chunk in simple_agent_graph.astream(inputs, stream_mode="updates"):
    for node, values in chunk.items():
        print(f"Receiving update from node: '{node}'")
        if node == "action":
          print(f"Tool Used: {values['messages'][0].name}")
        print(values["messages"])

        print("\n\n")

Receiving update from node: 'agent'
[AIMessage(content='', additional_kwargs={'tool_calls': [{'id': 'call_NNv27UINetfEXWQR5PfOag9v', 'function': {'arguments': '{"query": "A Comprehensive Survey of Deep Research"}', 'name': 'arxiv'}, 'type': 'function'}, {'id': 'call_14qxYwxBlRVYWwjaoo1trJrl', 'function': {'arguments': '{"query": "author of A Comprehensive Survey of Deep Research"}', 'name': 'tavily_search_results_json'}, 'type': 'function'}], 'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 60, 'prompt_tokens': 182, 'total_tokens': 242, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_name': 'gpt-4.1-nano-2025-04-14', 'system_fingerprint': 'fp_7c233bf9d1', 'id': 'chatcmpl-CKp9XnZvaA2JtabOdPe1FBttwhzBh', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None}, id='run--be3befdd-097

#### 🏗️ Activity #2:

Please write out the steps the agent took to arrive at the correct answer.

✅ Answer:

🧠 **Step-by-step analysis of what the agent did**

**Initial User Query**  
The agent received a question about the paper *“A Comprehensive Survey of Deep Research: Systems, Methodologies, and Applications.”*

**Planning Tool Calls**  
The agent decided it needed external sources.  
It generated two tool calls in parallel:  
- **Arxiv tool** → query `"A Comprehensive Survey of Deep Research"`  
- **Tavily search** → query `"author of A Comprehensive Survey of Deep Research"`

**Tool Execution**  
- Arxiv tool returned the metadata of the paper (title, publication date, authors: *Renjun Xu* and *Jingwen Peng*).  
- Tavily search provided additional references confirming the paper and its authors.

**Synthesizing Interim Answer**  
With the results, the agent responded:  
*“Found the paper authored by Renjun Xu and Jingwen Peng.”*  

It then reasoned that the user wanted more detail: *“Where do these authors work now?”*  
The agent triggered **follow-up tool calls** to answer this.

**Follow-up Tavily Calls**  
- Queried **Renjun Xu** → found profile confirming he is a *Principal Researcher at Zhejiang University*.  
- Queried **Jingwen Peng** → identified she is a *Lead Analyst at Liberty Mutual Investment Management* in Boston, MA.

**Final Synthesis**  
The agent combined these tool results and delivered a structured final answer:  
- *Renjun Xu → Zhejiang University*  
- *Jingwen Peng → Liberty Mutual Investment Management*



**summary:**  
This illustrates LangGraph’s cyclic agent loop: **plan → tool calls → refine → synthesize**.


⚙️ **Engineering Note #1: Parallel Calls**  
“Ideally, the agent should have first extracted the authors from Arxiv, then selectively queried Tavily for their current affiliations. Calling both in parallel was faster, but also redundant — a sequential dependency chain would have been more efficient for this well-scoped query.”  

But here’s why it didn’t wait:  
- The planner (the LLM inside the graph) wasn’t certain if Arxiv alone would yield enough author context.  
- It “hedged its bets” by also calling Tavily in parallel to try to retrieve author information right away.  
- This is common when an agent’s prompt encourages redundancy or *“use multiple tools if unsure.”*


⚙️ **Engineering Note #2: Sequential Follow-ups**  
For the second phase (finding current affiliations), the agent switched to **sequential Tavily calls**:  
- First for *Renjun Xu*  
- Then for *Jingwen Peng*  

This makes sense, because:  
- Each query depended on knowing the author’s **name first** (a dependency).  
- Executing them one by one allowed the agent to structure the search properly.  
- Running them in parallel here would not add much benefit — the cost is similar, and sequential execution avoids overloading the search tool with multiple requests at once.

📊 **Parallel vs Sequential Execution — Trade-offs**

| Strategy      | Advantages | Disadvantages | When to Use |
|---------------|------------|---------------|-------------|
| **Parallel**  | - Faster (lower latency)<br>- Redundancy/cross-checking<br>- Useful when tasks are independent | - More cost<br>- Adds noise<br>- Can be redundant | When tasks are independent and time-sensitive |
| **Sequential** | - Lower cost<br>- Cleaner dependency chain<br>- Less risk of noise | - Slower overall latency | When outputs depend on each other or query is well-scoped |

**Takeaway:**  
The agent’s behavior showed a mix of strategies: *parallelism* at the start to hedge uncertainty, then *sequential calls* when dependencies were clear.


# 🤝 Breakout Room #2

## Part 1: LangSmith Evaluator

### Pre-processing for LangSmith

To do a little bit more preprocessing, let's wrap our LangGraph agent in a simple chain.

In [17]:
def convert_inputs(input_object):
  return {"messages" : [HumanMessage(content=input_object["text"])]}

def parse_output(input_state):
  return {"answer" : input_state["messages"][-1].content}

agent_chain_with_formatting = convert_inputs | simple_agent_graph | parse_output

agent_chain_with_formatting.invoke({"text" : "What is Deep Research?"})

{'answer': 'Deep Research typically refers to an in-depth and comprehensive investigation or analysis into a specific subject or field. It involves thorough data collection, critical evaluation, and detailed examination to uncover insights, understand complex issues, or develop new knowledge. Deep Research is often used in academic, scientific, technological, and business contexts to inform decision-making, innovation, and strategic planning. \n\nIf you are referring to a specific organization, product, or platform named "Deep Research," please provide more context so I can give a more precise answer.'}

### Task 1: Creating An Evaluation Dataset

Just as we saw last week, we'll want to create a dataset to test our Agent's ability to answer questions.

In order to do this - we'll want to provide some questions and some answers. Let's look at how we can create such a dataset below.

```python
questions = [
    {
        "inputs" : {"text" : "Who were the main authors on the 'A Comprehensive Survey of Deep Research: Systems, Methodologies, and Applications' paper?"},
        "outputs" : {"must_mention" : ["Peng", "Xu"]}   
    },
    ...,
    {
        "inputs" : {"text" : "Where do the authors of the 'A Comprehensive Survey of Deep Research: Systems, Methodologies, and Applications' work now?"},
        "outputs" : {"must_mention" : ["Zhejiang", "Liberty Mutual"]}
    }
]
```

#### 🏗️ Activity #3:

Please create a dataset in the above format with at least 5 questions that pertain to the cohort use-case (more information [here](https://www.notion.so/Session-4-RAG-with-LangGraph-OSS-Local-Models-Eval-w-LangSmith-26acd547af3d80838d5beba464d7e701#26acd547af3d81d08809c9c82a462bdd)), or the use-case you're hoping to tackle in your Demo Day project.

In [18]:
questions = [
    # --- 4 business-style ---
    {
        "inputs": {
            "text": "What are the best AI use cases for law firms to save time and reduce costs?"
        },
        "outputs": {
            "must_mention": ["contract review", "compliance", "automation", "legal"]
        }
    },
    {
        "inputs": {
            "text": "How can healthcare providers use AI agents to improve patient care and efficiency?"
        },
        "outputs": {
            "must_mention": ["patient records", "summarization", "guidelines", "efficiency"]
        }
    },
    {
        "inputs": {
            "text": "What AI applications should banks prioritize to maximize ROI and stay compliant?"
        },
        "outputs": {
            "must_mention": ["fraud detection", "compliance", "regulations", "ROI"]
        }
    },
    {
        "inputs": {
            "text": "How can e-commerce companies leverage AI to boost sales and improve customer experience?"
        },
        "outputs": {
            "must_mention": ["recommendations", "chatbots", "inventory", "e-commerce"]
        }
    },

    # --- 4 depth ---
    {
        "inputs": {
            "text": "What specific tasks could an AI tutoring agent handle to support student learning?"
        },
        "outputs": {
            "must_mention": ["personalized learning", "assignments", "textbooks", "support"]
        }
    },
    {
        "inputs": {
            "text": "How could real estate buyers benefit from an AI assistant trained on property data?"
        },
        "outputs": {
            "must_mention": ["listings", "zoning", "mortgage", "pricing"]
        }
    },
    {
        "inputs": {
            "text": "What productivity gains could HR teams achieve by using AI in recruitment?"
        },
        "outputs": {
            "must_mention": ["resume screening", "candidate matching", "hiring", "efficiency"]
        }
    },
    {
        "inputs": {
            "text": "What problem in scientific research could an AI literature review agent solve?"
        },
        "outputs": {
            "must_mention": ["summarize papers", "find gaps", "research", "knowledge"]
        }
    },

    # --- 2 risk/ethics ---
    {
        "inputs": {
            "text": "What ethical concerns should healthcare startups address when using AI on patient data?"
        },
        "outputs": {
            "must_mention": ["privacy", "HIPAA", "consent", "bias"]
        }
    },
    {
        "inputs": {
            "text": "What risks should HR consider when relying on AI for hiring decisions?"
        },
        "outputs": {
            "must_mention": ["bias", "fairness", "discrimination", "accountability"]
        }
    }
]


Now we can add our dataset to our LangSmith project using the following code which we saw last Thursday!

In [19]:
from langsmith import Client

client = Client()

dataset_name = f"Simple Search Agent - Evaluation Dataset - {uuid4().hex[0:8]}"

dataset = client.create_dataset(
    dataset_name=dataset_name,
    description="Questions about the cohort use-case to evaluate the Simple Search Agent."
)

client.create_examples(
    dataset_id=dataset.id,
    examples=questions
)

{'example_ids': ['28fb3aa2-12ff-421b-ba06-2094fe10c4ba',
  'e539eb15-a3a9-46a8-9836-fb583cdd23f9',
  '75119d69-2197-4e2c-a12c-d66fd6f3f351',
  'da6660f9-c93e-4c45-963d-6b6e01363d6a',
  '454d6945-1b75-416d-8822-40e0e7970c8d',
  '7c74c481-5273-4a08-bde3-08a221b58618',
  '3b04b1ab-2759-4185-a7bc-6b9c109402de',
  '826a4688-26a6-4de9-973b-32e0e2612cb0',
  'ed05a8b6-6360-433a-94c9-5c7fd17f7492',
  '1412fd00-5144-42d2-afaf-3005d4847df9'],
 'count': 10}

### Task 2: Adding Evaluators

Let's use the OpenEvals library to product an evaluator that we can then pass into LangSmith!

> NOTE: Examine the `CORRECTNESS_PROMPT` below!

In [20]:
from openevals.prompts import CORRECTNESS_PROMPT
print(CORRECTNESS_PROMPT)

You are an expert data labeler evaluating model outputs for correctness. Your task is to assign a score based on the following rubric:

<Rubric>
  A correct answer:
  - Provides accurate and complete information
  - Contains no factual errors
  - Addresses all parts of the question
  - Is logically consistent
  - Uses precise and accurate terminology

  When scoring, you should penalize:
  - Factual errors or inaccuracies
  - Incomplete or partial answers
  - Misleading or ambiguous statements
  - Incorrect terminology
  - Logical inconsistencies
  - Missing key information
</Rubric>

<Instructions>
  - Carefully read the input and output
  - Check for factual accuracy and completeness
  - Focus on correctness of information rather than style or verbosity
</Instructions>

<Reminder>
  The goal is to evaluate factual correctness and completeness of the response.
</Reminder>

<input>
{inputs}
</input>

<output>
{outputs}
</output>

Use the reference outputs below to help you evaluate the

In [21]:
from openevals.llm import create_llm_as_judge

correctness_evaluator = create_llm_as_judge(
        prompt=CORRECTNESS_PROMPT,
        model="openai:o3-mini", # very impactful to the final score
        feedback_key="correctness",
    )

Let's also create a custom Evaluator for our created dataset above - we do this by first making a simple Python function!

In [22]:
def must_mention(inputs: dict, outputs: dict, reference_outputs: dict) -> float:
  # determine if the phrases in the reference_outputs are in the outputs
  required = reference_outputs.get("must_mention") or []
  score = all(phrase in outputs["answer"] for phrase in required)
  return score

#### ❓ Question #4:

What are some ways you could improve this metric as-is?

> NOTE: Alternatively you can suggest where gaps exist in this method.

✅ Answer:

### Current Implementation

The baseline `must_mention` function is:

```python
def must_mention(inputs: dict, outputs: dict, reference_outputs: dict) -> float:
    # determine if the phrases in the reference_outputs are in the outputs
    required = reference_outputs.get("must_mention") or []
    score = all(phrase in outputs["answer"] for phrase in required)
    return score
```

**Strength**: Simple sanity check — verifies whether key terms appear in the answer.

**Limitation**: Too brittle. Only works for exact string matches, binary output (0 or 1), no context sensitivity.

##### Major Gaps and Improvement Opportunities

### 1. __Binary Scoring Problem__

__Current Issue__: Returns only 0.0 or 1.0 (all-or-nothing)

- If 4 out of 5 required phrases are present, score = 0.0
- No partial credit for partially correct answers

__Improvement__:
We should return a proportion instead as a partial credit instead of binary responses 0 and 1.

```python
def must_mention_improved(inputs: dict, outputs: dict, reference_outputs: dict) -> float:
    required = reference_outputs.get("must_mention") or []
    if not required:
        return 1.0
    
    found_count = sum(1 for phrase in required if phrase.lower() in outputs["answer"].lower())
    return found_count / len(required)  # Partial scoring
```

### 2. __Case Sensitivity Issues__

__Current Issue__: "HIPAA" vs "hipaa" would fail to match. "Harvard" and "harvard" will also not match. 

__Improvement__: Use case-insensitive matching with `.lower()` 

```python
def must_mention_case_insensitive(inputs: dict, outputs: dict, reference_outputs: dict) -> float:
    required = reference_outputs.get("must_mention") or []
    answer = outputs["answer"].lower()
    return all(phrase.lower() in answer for phrase in required)
```

### 3. __Fuzzy Matching / Synonyms__

__Current Issue__:

- "machine learning" won't match "ML" or "machine-learning"
- "fraud detection" won't match "detecting fraud"
- "compliance checks" won't match "compliance"

__Improvements__:

- __Fuzzy matching__: Use libraries like `fuzzywuzzy` or `rapidfuxx` for approximate matches

```python
from rapidfuzz import fuzz

def must_mention_fuzzy(inputs: dict, outputs: dict, reference_outputs: dict, threshold: int = 80) -> float:
    required = reference_outputs.get("must_mention") or []
    answer = outputs["answer"].lower()
    
    def is_fuzzy_match(phrase, text):
        return fuzz.partial_ratio(phrase.lower(), text) >= threshold
    
    matches = sum(is_fuzzy_match(phrase, answer) for phrase in required)
    return matches / len(required) if required else 1.0

```

### 4. Stemming / Lemmatization

__Current Issue__: Doesn't account for:
- Small variations like "students" vs "student" should count as matches.
- Plural vs singular ("regulation" vs "regulations")
- Different word forms ("comply" vs "compliance")

__Improvement__: Use lemmatization or stemming - Normalize words using lemmatization.

```python
import nltk
from nltk.stem import WordNetLemmatizer
nltk.download("punkt", quiet=True)
nltk.download("wordnet", quiet=True)

lemmatizer = WordNetLemmatizer()

def must_mention_lemmatized(inputs: dict, outputs: dict, reference_outputs: dict) -> float:
    required = reference_outputs.get("must_mention") or []
    tokens = nltk.word_tokenize(outputs["answer"].lower())
    lemmas = {lemmatizer.lemmatize(tok) for tok in tokens}
    
    matches = sum(any(lemmatizer.lemmatize(word.lower()) in lemmas for word in phrase.split())
                  for phrase in required)
    return matches / len(required) if required else 1.0

```

### Advanced Enhancements:

#### 1. Weighting

__Current Issue__:
- Current Issue: All required phrases treated equally.

__Improvment__:

- Some terms are more important than others. Allow different weights.
- Example: "compliance" may be worth 2× "efficiency".

```python
def must_mention_weighted(inputs: dict, outputs: dict, reference_outputs: dict, weights=None) -> float:
    required = reference_outputs.get("must_mention") or []
    weights = weights or {phrase: 1.0 for phrase in required}
    total_weight = sum(weights.values())
    achieved = sum(weights[phrase] for phrase in required if phrase.lower() in outputs["answer"].lower())
    return achieved / total_weight if total_weight else 1.0

```

#### 2. Semantic Similarity
- Instead of only string-based checks, use embeddings to verify conceptual overlap or conceptual similarity.
``` python
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity

def must_mention_semantic(inputs: dict, outputs: dict, reference_outputs: dict, threshold=0.7) -> float:
    required = reference_outputs.get("must_mention") or []
    if not required:
        return 1.0
    
    model = SentenceTransformer("all-MiniLM-L6-v2")
    answer = outputs["answer"]
    answer_embedding = model.encode([answer])
    phrase_embeddings = model.encode(required)
    
    sims = cosine_similarity(phrase_embeddings, answer_embedding).flatten()
    return sum(1 for sim in sims if sim >= threshold) / len(required)

```

### 3. Context Awareness

__Current Issue__: Could match phrases in irrelevant contexts

- "bias" in "The study shows no bias" vs "AI systems can exhibit bias"

__Improvement__: Use NLP techniques to verify contextual relevance.

- Detect negations or misleading mentions ("not from Harvard" vs "Harvard").
- Could use dependency parsing or LLM-as-judge for nuanced scoring.


## Comprehensive Improved Version

```python
from fuzzywuzzy import fuzz
import re
from typing import List, Dict, Any

def must_mention_enhanced(
    inputs: dict, 
    outputs: dict, 
    reference_outputs: dict,
    fuzzy_threshold: int = 80,
    weights: Dict[str, float] = None
) -> float:
    """
    Enhanced must_mention evaluator with fuzzy matching, case insensitivity,
    partial scoring, and optional weighting.
    """
    required = reference_outputs.get("must_mention") or []
    if not required:
        return 1.0
    
    answer = outputs["answer"].lower()
    weights = weights or {phrase: 1.0 for phrase in required}
    
    total_weight = sum(weights.get(phrase, 1.0) for phrase in required)
    achieved_weight = 0.0
    
    for phrase in required:
        phrase_weight = weights.get(phrase, 1.0)
        phrase_lower = phrase.lower()
        
        # Exact match (highest score)
        if phrase_lower in answer:
            achieved_weight += phrase_weight
        # Fuzzy match (partial score)
        elif any(fuzz.partial_ratio(phrase_lower, word) >= fuzzy_threshold 
                for word in answer.split()):
            achieved_weight += phrase_weight * 0.8  # Reduced score for fuzzy match
    
    return achieved_weight / total_weight

# Alternative: Semantic similarity approach
def must_mention_semantic(inputs: dict, outputs: dict, reference_outputs: dict) -> float:
    """Uses sentence embeddings to check semantic similarity"""
    from sentence_transformers import SentenceTransformer
    
    required = reference_outputs.get("must_mention") or []
    if not required:
        return 1.0
    
    model = SentenceTransformer('all-MiniLM-L6-v2')
    answer = outputs["answer"]
    
    # Get embeddings
    answer_embedding = model.encode([answer])
    phrase_embeddings = model.encode(required)
    
    # Calculate similarities
    from sklearn.metrics.pairwise import cosine_similarity
    similarities = cosine_similarity(phrase_embeddings, answer_embedding).flatten()
    
    # Count phrases with similarity above threshold
    threshold = 0.7
    matches = sum(1 for sim in similarities if sim >= threshold)
    
    return matches / len(required)
```

## Additional Evaluation Metrics to Consider

### 1. __Completeness Score__

```python
def completeness_score(inputs: dict, outputs: dict, reference_outputs: dict) -> float:
    """Measures how completely the answer addresses the question"""
    # Could use LLM-as-judge for this
    pass
```

### 2. __Relevance Score__

```python
def relevance_score(inputs: dict, outputs: dict, reference_outputs: dict) -> float:
    """Measures how relevant the answer is to the question"""
    # Semantic similarity between question and answer
    pass
```

### 3. __Factual Accuracy__

```python
def factual_accuracy(inputs: dict, outputs: dict, reference_outputs: dict) -> float:
    """Checks factual correctness against known ground truth"""
    pass
```

**Note on Current Implementation**:
- An answer could just dump the words "Harvard, OpenAI, NBER" with no context. It would score 1 but clearly isn’t a good response.
- The metric doesn’t check correctness, only surface-level mention.
- If the agent adds incorrect info along with the correct mentions, it still scores 1.
- Exact-match approach makes the evaluation fragile → small wording differences drop the score to 0.

⚠️ Gaps in the Current Method

- Ignores answer quality (dumping keywords still passes).
- Ignores hallucinations (extra wrong info doesn’t reduce score).
- Too brittle (exact phrases only).
- No context check (negated or irrelevant mentions pass).


## Best Practices for Evaluation Metrics

1. __Use Multiple Metrics__: Combine `must_mention` with other metrics for comprehensive evaluation
2. __Domain-Specific Tuning__: Adjust thresholds and weights based on your specific use case
3. __Human Validation__: Regularly validate automated scores against human judgment
4. __Continuous Improvement__: Monitor metric performance and refine based on real-world usage
5. __Transparency__: Make evaluation criteria clear and explainable

## Summary

The current `must_mention` metric is a good starting point but has significant limitations. The main improvements needed are:

- __Partial scoring__ instead of binary
- __Case-insensitive matching__
- __Fuzzy/semantic matching__ for variations
- __Weighted importance__ for different phrases
- __Context awareness__ to avoid false positives

These improvements would make the evaluation more robust, fair, and aligned with human judgment of answer quality.


Task 3: Evaluating

All that is left to do is evaluate our agent's response!

In [23]:
results = client.evaluate(
    agent_chain_with_formatting,
    data=dataset.name,
    evaluators=[correctness_evaluator, must_mention],
    experiment_prefix="simple_agent, baseline",  # optional, experiment name prefix
    description="Testing the baseline system.",  # optional, experiment description
    max_concurrency=4, # optional, add concurrency
)

View the evaluation results for experiment: 'simple_agent, baseline-cf33c257' at:
https://smith.langchain.com/o/cbfba347-c08c-4a28-9b93-3cfe2f3af30b/datasets/f0b4d881-3618-40a9-a1d7-ba49bed1b533/compare?selectedSessions=4f01585b-51c5-459b-b400-3c18e319fddd




0it [00:00, ?it/s]

## Part 2: LangGraph with Helpfulness:

### Task 3: Adding Helpfulness Check and "Loop" Limits

Now that we've done evaluation - let's see if we can add an extra step where we review the content we've generated to confirm if it fully answers the user's query!

We're going to make a few key adjustments to account for this:

1. We're going to add an artificial limit on how many "loops" the agent can go through - this will help us to avoid the potential situation where we never exit the loop.
2. We'll add to our existing conditional edge to obtain the behaviour we desire.

First, let's define our state again - we can check the length of the state object, so we don't need additional state for this.

In [24]:
class AgentState(TypedDict):
  messages: Annotated[list, add_messages]

Now we can set our graph up! This process will be almost entirely the same - with the inclusion of one additional node/conditional edge!

#### 🏗️ Activity #4:

Please write markdown for the following cells to explain what each is doing.

##### MARKDOWN #1

✅ Answer:
### Setting Up the Enhanced Graph with Helpfulness Validation

We initializes a new `StateGraph` instance that will include helpfulness checking capabilities. It’s structurally similar to the earlier graph but adds the foundation for later validation logic.

- **`graph_with_helpfulness_check`**: A new StateGraph built on the same `AgentState` schema, which stores the evolving conversation state..
- **`"agent"` node**: Runs the call_model function. This is the LLM node that processes user queries, reasons about next steps, and decides whether to continue, stop, or call a tool.
- **`"action"` node**: Runs the tool_node function. This is the tool execution node that actually performs external lookups (e.g., Tavily, Arxiv, etc) when requested by the agent.

Together, these two nodes form the backbone of the agent’s reasoning + action cycle.

In [25]:
graph_with_helpfulness_check = StateGraph(AgentState)

graph_with_helpfulness_check.add_node("agent", call_model)
graph_with_helpfulness_check.add_node("action", tool_node)

##### MARKDOWN #2

✅ Answer:

### Defining the Graph Entry Point

This cell sets the entry point for our enhanced graph to the `"agent"` node. This means:

- When the graph starts execution, it will first call the `call_model` function
- The agent node will receive the initial user query and begin processing
- This is the same entry point as our previous graph, maintaining consistency in the execution flow

The entry point determines where the conversation begins in our agent workflow.



In [27]:
graph_with_helpfulness_check.set_entry_point("agent")

##### MARKDOWN #3

✅ Answer:

### Implementing Smart Routing with Helpfulness Validation

This cell defines the core decision function, `tool_call_or_helpful`, which introduces intelligent routing logic to our graph. It decides whether the agent should call a tool, continue refining its answer, or stop execution.

The routing works in three layers:

**1. Tool Call Detection (Priority #1)**
- Checks if the last message contains `tool_calls`
- If tools are needed, the graph routes to the `"action"` node so the required tool can be executed.

**2. Loop Prevention (Safety Mechanism)**
- Monitors the total message count: 
- If more than 10 messages are present, it forces an `"END"` to prevent infinite loops and resource overuse.
**3. Helpfulness Evaluation (Quality Control)**
- Creates a separate LLM chain using `gpt-4.1-mini` to evaluate response quality

- Compares the initial user query against the final agent response

- Uses a simple prompt asking for Y/N helpfulness assessment
    - **If helpful ("Y")**: Routes to `"end"` → terminates successfully
    - **If unhelpful ("N")**: Routes to `"continue"` → loops back to agent for improvement

This creates a **self-improving feedback loop** where the agent can refine its responses until they meet quality standards.



In [31]:
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import StrOutputParser

def tool_call_or_helpful(state):
  last_message = state["messages"][-1]

  if last_message.tool_calls:
    return "action"

  initial_query = state["messages"][0]
  final_response = state["messages"][-1]

  if len(state["messages"]) > 10:
    return "END"

  prompt_template = """\
  Given an initial query and a final response, determine if the final response is extremely helpful or not. Please indicate helpfulness with a 'Y' and unhelpfulness as an 'N'.

  Initial Query:
  {initial_query}

  Final Response:
  {final_response}"""

  helpfullness_prompt_template = PromptTemplate.from_template(prompt_template)

  helpfulness_check_model = ChatOpenAI(model="gpt-4.1-mini")

  helpfulness_chain = helpfullness_prompt_template | helpfulness_check_model | StrOutputParser()

  helpfulness_response = helpfulness_chain.invoke({"initial_query" : initial_query.content, "final_response" : final_response.content})

  if "Y" in helpfulness_response:
    return "end"
  else:
    return "continue"

##### MARKDOWN #4
✅ Answer:

### Mapping Routing Decisions to Graph Nodes

This cell configures the conditional edge routing by mapping the outputs of `tool_call_or_helpful` to specific graph destinations:

- **`"continue"`** → `"agent"`: Loops back to the agent node so it can refine its response.
- **`"action"`** → `"action"`: Executes the appropriate tool when the agent requests them. 
- **`"end"`** → `END`: Terminates the graph when the response is deemed helpful.

This mapping enables three distinct execution paths:
1. **Tool execution path**: When external data is needed
2. **Refinement path**: When response quality is insufficient
3. **Completion path**: When response meets helpfulness criteria

The routing creates a dynamic workflow that adapts based on both tool requirements and response quality.


In [32]:
graph_with_helpfulness_check.add_conditional_edges(
    "agent",
    tool_call_or_helpful,
    {
        "continue" : "agent",
        "action" : "action",
        "end" : END
    }
)

Adding an edge to a graph that has already been compiled. This will not be reflected in the compiled graph.


##### MARKDOWN #5

✅ Answer:

### Establishing Tool Result Processing Flow

This cell creates a direct edge from the `"action"` node back to the `"agent"` node: `graph_with_helpfulness_check.add_edge("action", "agent")`
This ensures:

- **Seamless tool integration**: After tools execute, results automatically flow back to the agent's reasoning loop.
- **Consistent processing**: The agent can incorporate tool results into its reasoning just like it does with user input.
- **Continued evaluation**: Tool results will be subject to the same helpfulness evaluation, ensuring quality.

This edge completes the tool execution cycle: Agent → Tools → Agent → Helpfulness Check.

This guarantees that external information is fully absorbed and refined before producing a final answer.

In [33]:
graph_with_helpfulness_check.add_edge("action", "agent")

Adding an edge to a graph that has already been compiled. This will not be reflected in the compiled graph.


##### MARKDOWN #6

✅ Answer:

### Compiling the Enhanced Agent Graph

This cell compiles our enhanced graph into an executable workflow: `agent_with_helpfulness_check = graph_with_helpfulness_check.compile()`

The compilation process perform three key functions:

- **Validates graph structure**: Ensures all nodes and edges are properly connected
- **Optimizes execution**: Prepares the graph for efficient runtime performance
- **Creates executable instance**: `agent_with_helpfulness_check` becomes a **runnable** agent

By compiling, we now have a complete agent that integrates both:

- Tool-calling capabilities (retrieving external knowledge)
- Helpfulness validation (automatic quality assessment)
- This makes the system not just functional, but robust and self-improving.


In [34]:
agent_with_helpfulness_check = graph_with_helpfulness_check.compile()

##### MARKDOWN #7

✅ Answer:

### Demonstrating Enhanced Agent Capabilities

In this cell, we test our enhanced agent with a sample query about "Deep Research Agents". The graph is executed in streaming mode, which allows us to observe its step-by-step reasoning.
The execution will demonstrate:

**Enhanced Workflow Features:**
- **Tool utilization**: Agent may call Tavily/Arxiv fto fetch external information
- **Quality assessment**: Each response undergoes helpfulness check
- **Iterative refinement**: Agent continues improving until response meets quality standards
- **Automatic termination**: Graph stops or Execution ends once the response is deemed sufficiently helpful

**Streaming Output Benefits:**
- **Real-time visibility**: See each node execution as it happens
- **Debugging capability**: Monitor tool calls, responses, and routing decisions
- **Performance insights**: Understand how many cycles the agent requires

By combining tool integration with automatic response validation, this test demonstrates a production-ready agent that balances thoroughness with efficiency through intelligent quality control.



In [35]:
inputs = {"messages" : [HumanMessage(content="What are Deep Research Agents?")]}

async for chunk in agent_with_helpfulness_check.astream(inputs, stream_mode="updates"):
    for node, values in chunk.items():
        print(f"Receiving update from node: '{node}'")
        print(values["messages"])
        print("\n\n")

Receiving update from node: 'agent'
[AIMessage(content='Deep Research Agents are advanced AI systems designed to assist with in-depth research tasks. They leverage deep learning techniques and large datasets to analyze complex information, generate insights, and support decision-making across various fields such as science, technology, medicine, and more. These agents can automate literature reviews, extract relevant data from vast sources, and provide comprehensive summaries, making research processes more efficient and thorough. Would you like me to find more detailed or specific information about Deep Research Agents?', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 94, 'prompt_tokens': 158, 'total_tokens': 252, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_name': 'gpt-4.1-

## Part 3: LangGraph for the "Patterns" of GenAI

### Task 4: Helpfulness Check of Gen AI Pattern Descriptions

Let's ask our system about the 3 main patterns in Generative AI:

1. Context Engineering
2. Fine-tuning
3. Agents

In [36]:
patterns = ["Context Engineering", "Fine-tuning", "LLM-based agents"]

In [37]:
for pattern in patterns:
  what_is_string = f"What is {pattern} and when did it break onto the scene??"
  inputs = {"messages" : [HumanMessage(content=what_is_string)]}
  messages = agent_with_helpfulness_check.invoke(inputs)
  print(messages["messages"][-1].content)
  print("\n\n")

Context Engineering is a relatively new interdisciplinary field that focuses on designing, managing, and optimizing the context in which systems, especially artificial intelligence and software applications, operate. It involves understanding and shaping the environment, circumstances, and background information that influence how systems behave and interact with users. The goal is to improve system performance, user experience, and decision-making by carefully engineering the context.

The concept of Context Engineering has gained prominence with the rise of AI and ubiquitous computing, where the context significantly impacts system effectiveness. It started to break onto the scene in the early 2020s, particularly with advancements in AI, contextual computing, and human-computer interaction research. The field is still evolving, but it is increasingly recognized as crucial for developing more adaptive, intelligent, and user-centric systems.

Would you like more detailed information on